# Resource and channel success-rate forecasts

Six direct five-minute targets use only readings available at each forecast origin. Train the models from the repository root before running this notebook.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

REPORT = Path("reports/kpi_forecast")
summary = json.loads((REPORT / "summary.json").read_text())
assert summary["raw_train_rows"] == 8064 and summary["raw_test_rows"] == 2016
print("Targets:", ", ".join(summary["targets"]))
print("Training and test rows:", summary["raw_train_rows"], summary["raw_test_rows"])

## Which candidate was chosen?

The leaderboard is ranked by mean validation MAE, one target at a time. A persistence or trailing-mean baseline is eligible to win.

In [ ]:
board = pd.read_csv(REPORT / "cv_leaderboard.csv")
for target in summary["targets"]:
    group = board.loc[board.target.eq(target), ["candidate", "mean_cv_mae_pp", "mean_cv_rmse_pp"]]
    print("\n", target, "\n", group.head(3).to_string(index=False))

## Held-out test comparison

Scores below are in percentage points. The selected candidate was fixed using the training folds; it was not chosen from these test results.

In [ ]:
scores = pd.read_csv(REPORT / "test_metrics.csv")
selected = scores.loc[scores.selected].set_index("target")
persistence = scores.loc[scores.candidate.eq("persistence")].set_index("target")
compare = pd.DataFrame({"selected": selected.mae_pp, "persistence": persistence.mae_pp})
print(compare.round(4).to_string())
ax = compare.plot(kind="bar", figsize=(10, 4), rot=35)
ax.set_ylabel("Test MAE (percentage points)")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

## Check one saved inference record

At the first test origin, the target is exactly five minutes later. The saved prediction does not include the actual future observation.

In [ ]:
example = json.loads((REPORT / "cpu_utilization_inference.json").read_text())
print(json.dumps(example, indent=2))
assert example["target_timestamp"] == "2025-10-25T14:29:00"
assert "actual" not in example

## Reading the results

Trailing means won the three utilization targets; Ridge won the three payment success-rate targets on the training folds. The supplied test file follows training, but discovery already inspected the whole monitoring period. These are point forecasts; uncertainty and incident risk were not estimated.